In [7]:
%pip install requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [13]:
import requests   # Makes API calls to Google
import time       
import csv        
import json       
from dotenv import load_dotenv
import os

In [9]:
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
# loading API key which is stored in another file
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")

In [31]:
# filter by county
county = "Collin County"
state = "Texas"
search_location = f"{county}, {state}"
output_file = f"{county.lower().replace(' ', '_')}_farms.csv"

In [32]:
# search criteria (Used Professors Godat prompt)
queries = [ 
    "urban farm",
    "market garden",
    "hydroponic farm",
    "aquaponic farm",
    "greenhouse farm",
    "vertical farm",
    "indoor farm",
    "microgreens",
    "CSA farm",
    "flower farm",
    "backyard farm",
    "egg farm",
    "honey farm",
    "mushroom farm",
    "microfarm",
    "small farm",
    "community supported agriculture"
]
    

In [33]:
def search_places(query):
    url = "https://maps.googleapis.com/maps/api/place/textsearch/json"

    all_results = []

    params = {
        "query":f"{query} in {search_location}",
        "key": API_KEY
    }

    while True:
        print(f"  Fetching: {query}...")
        response = requests.get(url, params=params)

        data = response.json()
        all_results.extend(data.get("results", []))

        next_token = data.get("next_page_token")
        if not next_token:
            break

        time.sleep(2)
        params = {"pagetoken": next_token, "key": API_KEY}

    return all_results

In [18]:
def get_details(place_id):
    url = "https://maps.googleapis.com/maps/api/place/details/json"

    params = {
        "place_id": place_id,
        "fields": "name,formatted_address,geometry,website,types",
        "key": API_KEY
    }

    response = requests.get(url, params=params)
    return response.json().get("result", {})

In [34]:
def export_to_csv(places, filename=output_file):
    print(f"\nExporting {len(places)} places to {filename}...")

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "Name",
            "Address",
            "Latitude",
            "Longitude",
            "Website",
            "Google Categories",
            "Place ID",
            "Status"
        ])

        for place in places:
            print(f"  Getting details for: {place.get('name', 'Unknown')}...")
            details = get_details(place["place_id"])

            geometry = details.get("geometry", {})
            location = geometry.get("location", {})
            lat = location.get("lat", "")
            lng = location.get("lng", "")

            writer.writerow([
                details.get("name", ""),
                details.get("formatted_address", ""),
                lat,
                lng,
                details.get("website", "Not listed"),
                ", ".join(details.get("types", [])),
                place["place_id"],
                "Pending Review"
            ])

            time.sleep(0.5)

    print(f"File saved as: {filename}")

In [35]:
def main():
    all_places = {}

    for query in queries:
        print(f"\nSearching: '{query}' in {search_location}")
        results = search_places(query)
        print(f"  Found {len(results)} results")

        for place in results:
            place_id = place["place_id"]
            if place_id not in all_places:
                all_places[place_id] = place

        time.sleep(1)

    print(f"\nTotal unique places found: {len(all_places)}")
    export_to_csv(list(all_places.values()))

main()


Searching: 'urban farm' in Collin County, Texas
  Fetching: urban farm...
  Fetching: urban farm...
  Found 20 results

Searching: 'market garden' in Collin County, Texas
  Fetching: market garden...
  Found 0 results

Searching: 'hydroponic farm' in Collin County, Texas
  Fetching: hydroponic farm...
  Found 8 results

Searching: 'aquaponic farm' in Collin County, Texas
  Fetching: aquaponic farm...
  Found 5 results

Searching: 'greenhouse farm' in Collin County, Texas
  Fetching: greenhouse farm...
  Fetching: greenhouse farm...
  Found 20 results

Searching: 'vertical farm' in Collin County, Texas
  Fetching: vertical farm...
  Found 1 results

Searching: 'indoor farm' in Collin County, Texas
  Fetching: indoor farm...
  Found 18 results

Searching: 'microgreens' in Collin County, Texas
  Fetching: microgreens...
  Fetching: microgreens...
  Found 14 results

Searching: 'CSA farm' in Collin County, Texas
  Fetching: CSA farm...
  Found 17 results

Searching: 'flower farm' in Colli